<a href="https://colab.research.google.com/github/Innovatewithapple/Classification/blob/main/Classcification_Practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
from google.colab import userdata
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split,GridSearchCV,cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from xgboost import XGBClassifier

In [ ]:
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')

In [ ]:
!kaggle datasets download -d blastchar/telco-customer-churn

Dataset URL: https://www.kaggle.com/datasets/blastchar/telco-customer-churn
License(s): copyright-authors
100% 172k/172k [00:00<00:00, 97.8MB/s]



In [ ]:
!unzip -q /content/telco-customer-churn.zip -d /content/

In [ ]:
df = pd.read_csv('/content/WA_Fn-UseC_-Telco-Customer-Churn.csv')
df.sample(10)

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
4728,6119-SPUDB,Male,0,No,No,46,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,Two year,No,Mailed check,38.25,1755.35,No
1282,6260-ONULR,Male,0,No,No,1,Yes,No,DSL,No,...,No,No,Yes,Yes,Month-to-month,Yes,Mailed check,62.80,62.8,No
1832,3132-TVFDZ,Male,1,Yes,No,57,No,No phone service,DSL,No,...,No,No,Yes,Yes,Month-to-month,Yes,Electronic check,44.85,2572.95,Yes
6949,3648-GZPHF,Male,0,Yes,Yes,32,No,No phone service,DSL,No,...,Yes,Yes,No,No,One year,Yes,Mailed check,36.25,1151.05,No
6331,1929-ZCBHE,Male,0,Yes,Yes,47,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Electronic check,40.30,1794.8,No
811,4853-RULSV,Male,0,No,No,70,Yes,Yes,Fiber optic,Yes,...,No,Yes,Yes,Yes,Two year,Yes,Credit card (automatic),104.00,7250.15,Yes
2727,3387-VATUS,Male,0,No,No,5,Yes,Yes,Fiber optic,No,...,No,No,Yes,Yes,Month-to-month,Yes,Bank transfer (automatic),94.85,462.8,Yes
2419,2450-ZKEED,Female,0,No,No,11,Yes,No,DSL,No,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),53.80,651.55,No
2021,0181-RITDD,Male,0,Yes,Yes,62,Yes,No,Fiber optic,Yes,...,Yes,Yes,Yes,Yes,Two year,No,Mailed check,108.15,6825.65,No
1545,5193-QLVZB,Male,0,No,No,63,Yes,Yes,Fiber optic,No,...,Yes,No,Yes,Yes,Two year,Yes,Bank transfer (automatic),104.75,6536.5,No


In [ ]:
df.columns

Index(['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents',
       'tenure', 'PhoneService', 'MultipleLines', 'InternetService',
       'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
       'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling',
       'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn'],
      dtype='object')

In [ ]:
df.isnull().sum()

,0
customerID,0
gender,0
SeniorCitizen,0
Partner,0
Dependents,0
tenure,0
PhoneService,0
MultipleLines,0
InternetService,0
OnlineSecurity,0


In [ ]:
(df['TotalCharges'] == " ").sum()

np.int64(11)

In [ ]:
df['TotalCharges'].unique()

array(['29.85', '1889.5', '108.15', ..., '346.45', '306.6', '6844.5'],
      dtype=object)

In [ ]:
df['TotalCharges'] = df['TotalCharges'].replace(" ", np.nan)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'])
df['TotalCharges'].fillna(0,inplace=True)

/tmp/ipykernel_2274/698073160.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['TotalCharges'].fillna(0,inplace=True)


In [ ]:
x = df.drop(columns=['customerID','Churn'])
y = df['Churn']

In [ ]:
y = y.map({'Yes':1,'No':0})

In [ ]:
x.shape

(7043, 19)

In [ ]:
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.3,random_state=42)

In [ ]:
#get columns
num_cols = x.select_dtypes(include=['int64','float64']).columns
cat_cols = x.select_dtypes(include='object').columns

In [ ]:
#Preprocessing
preprocessor = ColumnTransformer([
    ('nums',StandardScaler(),num_cols),
    ('cat',OneHotEncoder(drop='first'),cat_cols)
])

Logistic Pipeline

In [ ]:
logistic_pipeline = Pipeline([
    ('preprocess',preprocessor),
    ('model',LogisticRegression())
])

In [ ]:
logistic_pipeline.fit(x_train,y_train)

Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('nums', StandardScaler(),
                                                  Index(['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges'], dtype='object')),
                                                 ('cat',
                                                  OneHotEncoder(drop='first'),
                                                  Index(['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
       'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
       'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
       'PaperlessBilling', 'PaymentMethod'],
      dtype='object'))])),
                ('model', LogisticRegression())])

In [ ]:
logistic_cross_Score = cross_validate(logistic_pipeline,x_train,y_train,cv=5,scoring=['accuracy',"precision", "recall", "f1"])
logistic_cross_Score

{'fit_time': array([0.15785098, 0.17742038, 0.24877   , 0.24567771, 0.11262012]),
 'score_time': array([0.03142285, 0.08048081, 0.06392527, 0.0773046 , 0.03437543]),
 'test_accuracy': array([0.80628803, 0.81845842, 0.78803245, 0.79513185, 0.80831643]),
 'test_precision': array([0.65044248, 0.69607843, 0.61792453, 0.63507109, 0.68617021]),
 'test_recall': array([0.56756757, 0.54826255, 0.50579151, 0.51737452, 0.4980695 ]),
 'test_f1': array([0.60618557, 0.61339093, 0.55626327, 0.57021277, 0.57718121])}

In [ ]:
logistic_param_grid = [
    {
        # Path A: The "Executioner" (Killing useless features)
        'model__penalty': ['l1'],
        'model__C': [0.01, 0.1, 1, 10],
        'model__solver': ['liblinear'],  # liblinear is the pro choice for L1 + small binary data
        'model__class_weight': ['balanced', None] # Test our "magic button"
    },
    {
        # Path B: The "Gentle Shrinker" (Standard optimization)
        'model__penalty': ['l2'],
        'model__C': [0.01, 0.1, 1, 10],
        'model__solver': ['lbfgs'],       # Standard robust solver
        'model__class_weight': ['balanced', None]
    }
]

In [ ]:
grid = GridSearchCV(logistic_pipeline,param_grid=logistic_param_grid,cv=5,scoring='f1',verbose=0)
grid.fit(x_train,y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('preprocess',
                                        ColumnTransformer(transformers=[('nums',
                                                                         StandardScaler(),
                                                                         Index(['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges'], dtype='object')),
                                                                        ('cat',
                                                                         OneHotEncoder(drop='first'),
                                                                         Index(['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
       'InternetService', 'OnlineSecurity', 'OnlineBackup...
       'PaperlessBilling', 'PaymentMethod'],
      dtype='object'))])),
                                       ('model', LogisticRegression())]),
             param_grid=[{'model__C': [0.01, 0.1, 1, 10],
                          'model__class_weight': ['balanced', None],
                          'model__penalty': ['l1'],
                          'model__solver': ['liblinear']},
                         {'model__C': [0.01, 0.1, 1, 10],
                          'model__class_weight': ['balanced', None],
                          'model__penalty': ['l2'],
                          'model__solver': ['lbfgs']}],
             scoring='f1')

In [ ]:
print("Best Parameters:", grid.best_params_)
print("Best F1 Score:", grid.best_score_)

Best Parameters: {'model__C': 0.1, 'model__class_weight': 'balanced', 'model__penalty': 'l1', 'model__solver': 'liblinear'}
Best F1 Score: 0.6133148694265009


In [ ]:
y_pred = grid.best_estimator_.predict(x_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.92      0.73      0.81      1539
           1       0.53      0.83      0.65       574

    accuracy                           0.76      2113
   macro avg       0.73      0.78      0.73      2113
weighted avg       0.81      0.76      0.77      2113



Random-Forest

In [ ]:
random_forest_Pipeline = Pipeline([
    ('preprocessor',preprocessor),
    ('model',RandomForestClassifier())
])

In [ ]:
random_forest_Pipeline.fit(x_train,y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('nums', StandardScaler(),
                                                  Index(['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges'], dtype='object')),
                                                 ('cat',
                                                  OneHotEncoder(drop='first'),
                                                  Index(['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
       'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
       'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
       'PaperlessBilling', 'PaymentMethod'],
      dtype='object'))])),
                ('model', RandomForestClassifier())])

In [ ]:
random_forest_cross_score = cross_validate(random_forest_Pipeline,x_train,y_train,cv=5,scoring=['accuracy',"precision", "recall", "f1"])
random_forest_cross_score

{'fit_time': array([1.37206268, 1.14816022, 1.07967901, 0.65113378, 0.59688091]),
 'score_time': array([0.07479692, 0.07768488, 0.04711151, 0.04581404, 0.05449891]),
 'test_accuracy': array([0.79411765, 0.80121704, 0.78194726, 0.78194726, 0.78093306]),
 'test_precision': array([0.62962963, 0.67027027, 0.60784314, 0.61702128, 0.61375661]),
 'test_recall': array([0.52509653, 0.47876448, 0.47876448, 0.44787645, 0.44787645]),
 'test_f1': array([0.57263158, 0.55855856, 0.53563715, 0.51901566, 0.51785714])}

In [ ]:
random_grid_params = {
    "model__n_estimators":[100,200],
    "model__max_depth":[10,20,None],
    "model__min_samples_split":[2,5,20],
    "model__min_samples_leaf":[1,2,4],
    "model__max_features": ['sqrt', 'log2'],
    'model__class_weight': ['balanced']
}

In [ ]:
random_forest_gridsearch = GridSearchCV(random_forest_Pipeline,param_grid=random_grid_params,scoring='f1',verbose=0,cv=5)
random_forest_gridsearch.fit(x_train,y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('preprocessor',
                                        ColumnTransformer(transformers=[('nums',
                                                                         StandardScaler(),
                                                                         Index(['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges'], dtype='object')),
                                                                        ('cat',
                                                                         OneHotEncoder(drop='first'),
                                                                         Index(['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
       'InternetService', 'OnlineSecurity', 'OnlineBack...
       'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
       'PaperlessBilling', 'PaymentMethod'],
      dtype='object'))])),
                                       ('model', RandomForestClassifier())]),
             param_grid={'model__class_weight': ['balanced'],
                         'model__max_depth': [10, 20, None],
                         'model__max_features': ['sqrt', 'log2'],
                         'model__min_samples_leaf': [1, 2, 4],
                         'model__min_samples_split': [2, 5, 20],
                         'model__n_estimators': [100, 200]},
             scoring='f1')

In [ ]:
print('best params: ',random_forest_gridsearch.best_params_)
print('best score: ',random_forest_gridsearch.best_score_)
print('best estimator: ',random_forest_gridsearch.best_estimator_)

best params:  {'model__class_weight': 'balanced', 'model__max_depth': 10, 'model__max_features': 'log2', 'model__min_samples_leaf': 2, 'model__min_samples_split': 5, 'model__n_estimators': 200}
best score:  0.6299109355135755
best estimator:  Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('nums', StandardScaler(),
                                                  Index(['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges'], dtype='object')),
                                                 ('cat',
                                                  OneHotEncoder(drop='first'),
                                                  Index(['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
       'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
       'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
       'PaperlessBilling', 'PaymentMethod'],
      dtype='object'))])),
                ('model',
    

In [ ]:
# Use the best model to predict
y_pred_forest = random_forest_gridsearch.best_estimator_.predict(x_test)

# Import the professional report
from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(y_test, y_pred_forest))

              precision    recall  f1-score   support

           0       0.90      0.79      0.84      1539
           1       0.58      0.76      0.66       574

    accuracy                           0.78      2113
   macro avg       0.74      0.78      0.75      2113
weighted avg       0.81      0.78      0.79      2113



XGBoost

In [ ]:
xg_pipeline = Pipeline([
    ('preprocess',preprocessor),
    ('model',XGBClassifier(
        eval_metric='logloss',
        random_state=42,
    ))
])

In [ ]:
xg_pipeline.fit(x_train,y_train)

Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('nums', StandardScaler(),
                                                  Index(['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges'], dtype='object')),
                                                 ('cat',
                                                  OneHotEncoder(drop='first'),
                                                  Index(['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
       'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
       'TechSu...
                               feature_types=None, feature_weights=None,
                               gamma=None, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=None,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=None, max_leaves=None,
                               min_child_weight=None, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=None, n_jobs=None,
                               num_parallel_tree=None, ...))])

In [ ]:
xgb_param_grid = {
    'model__n_estimators': [100, 200],
    'model__learning_rate': [0.01, 0.1],
    'model__max_depth': [3, 5],
    'model__min_child_weight': [1, 5],    # The "Leaf" replacement
    'model__gamma': [0, 0.1, 0.2],         # The "Threshold" for a split
    'model__subsample': [0.8, 1.0],
    'model__colsample_bytree': [0.8, 1.0],
    'model__scale_pos_weight': [3]         # We keep it at 3 to focus on Recall
}

In [ ]:
xgb_gridsearch = GridSearchCV(xg_pipeline,param_grid=xgb_param_grid,scoring='f1',verbose=0,cv=5)
xgb_gridsearch.fit(x_train,y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('preprocess',
                                        ColumnTransformer(transformers=[('nums',
                                                                         StandardScaler(),
                                                                         Index(['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges'], dtype='object')),
                                                                        ('cat',
                                                                         OneHotEncoder(drop='first'),
                                                                         Index(['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
       'InternetService', 'OnlineSecurity', 'OnlineBackup...
                                                      multi_strategy=None,
                                                      n_estimators=None,
                                                      n_jobs=None,
                                                      num_parallel_tree=None, ...))]),
             param_grid={'model__colsample_bytree': [0.8, 1.0],
                         'model__gamma': [0, 0.1, 0.2],
                         'model__learning_rate': [0.01, 0.1],
                         'model__max_depth': [3, 5],
                         'model__min_child_weight': [1, 5],
                         'model__n_estimators': [100, 200],
                         'model__scale_pos_weight': [3],
                         'model__subsample': [0.8, 1.0]},
             scoring='f1')

In [ ]:
print('params: ',xgb_gridsearch.best_params_)
print('best score: ',xgb_gridsearch.best_score_)

params:  {'model__colsample_bytree': 0.8, 'model__gamma': 0.2, 'model__learning_rate': 0.1, 'model__max_depth': 5, 'model__min_child_weight': 5, 'model__n_estimators': 100, 'model__scale_pos_weight': 3, 'model__subsample': 0.8}
best score:  0.6258017749541096


In [ ]:
xgb_y_pred = xgb_gridsearch.best_estimator_.predict(x_test)

print(classification_report(y_test,xgb_y_pred))

              precision    recall  f1-score   support

           0       0.91      0.74      0.82      1539
           1       0.53      0.79      0.64       574

    accuracy                           0.76      2113
   macro avg       0.72      0.77      0.73      2113
weighted avg       0.80      0.76      0.77      2113

